## Scraper de buscametas.com/clasificaciones/

A diferencia de mychip.es y carreirasgalegas.com, aquí **no hay API/SPA** detrás del calendario: la página es HTML estático normal. Cada evento es una ficha `.crono-item` con fecha, nombre, ubicación y una o varias modalidades (cada una con un `codigo`). No se ha encontrado paginación ni filtro por año — con 147 eventos / 326 códigos de modalidad, parece ser todo el histórico que enseña el sitio.

Los **resultados** de cada modalidad se cargan vía AJAX con dos POST:
- `get_dorsales.php` → un registro por dorsal inscrito (tenga tiempo o no): nombre, sexo, club, categoría, estado...
- `get_tiempos.php` → un registro solo para los dorsales con cronometraje: tiempo, inicio, fin, parciales...

El campo `posicion` de la API siempre viene `null` — el ranking (general, por sexo, por categoría) lo calcula el propio script ordenando los tiempos (función `build_classification`), igual que hace el Javascript de la página.

**Riesgo no verificado**: las llamadas se probaron desde el navegador, no se pudo confirmar que `requests` no sea bloqueado por fingerprinting. Si aparecen errores 403/404 persistentes (no solo 502/503/504 transitorios), puede haber que añadir cabeceras o pasar a Selenium/Playwright.

**Cómo ejecutarlo**: `pip install requests beautifulsoup4` y luego `python buscametas_scraper.py` (o `main()` en una celda de Jupyter). Soporta pausar/reanudar (`RESUME`) y viene con `MAX_EVENTS=5` de fábrica para una prueba pequeña antes de lanzar todo el histórico.

In [3]:
# Scraper de buscametas.com/clasificaciones/ — ver el resumen en la
# celda markdown de arriba (que hace, que APIs usa, como ejecutarlo).

from pathlib import Path
import csv
import json
import os
import time

import requests
from bs4 import BeautifulSoup

# ---------------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------------
BASE_URL = "https://www.buscametas.com"
CALENDAR_URL = f"{BASE_URL}/clasificaciones/"
DORSALES_URL = f"{BASE_URL}/modulos/clasif/fuentes/get_dorsales.php"
TIEMPOS_URL = f"{BASE_URL}/modulos/clasif/fuentes/get_tiempos.php"

BROWSER_USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
)
HTML_HEADERS = {
    "User-Agent": BROWSER_USER_AGENT,
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "es-ES,es;q=0.9,ca;q=0.8,en;q=0.7",
}

SUBCONJUNTO = 0  # segon parametre que demanen els endpoints; "0" a totes les proves fetes

MAX_EVENTS = None   # posa None per no limitar (~147 esdeveniments)

SLEEP_BETWEEN_REQUESTS = 0.5  # segons; per no bombardejar el lloc
MAX_RETRIES = 8
RETRY_BACKOFF_SECONDS = 3
RETRY_BACKOFF_MAX = 30

SAVE_EVERY_N_EVENTS = 10

OUTPUT_DIR = Path("../../data/raw/buscametas/buscametas_output")
CURSES_CSV = os.path.join(OUTPUT_DIR, "curses.csv")
CLASSIF_CSV = os.path.join(OUTPUT_DIR, "classificacions.csv")
CHECKPOINT_FILE = os.path.join(OUTPUT_DIR, "buscametas_checkpoint.json")

# Pausar i reprendre: igual que amb mychip.es/carreirasgalegas.com — per
# defecte CONTINUA on ho havies deixat en lloc de tornar a començar. Esborra
# OUTPUT_DIR (o posa RESUME = False) per forçar un comencament de zero.
RESUME = True

_MESOS = {
    "ENE": "01", "FEB": "02", "MAR": "03", "ABR": "04", "MAY": "05", "JUN": "06",
    "JUL": "07", "AGO": "08", "SEP": "09", "OCT": "10", "NOV": "11", "DIC": "12",
}

_session = requests.Session()


# ---------------------------------------------------------------------------
# CRIDES HTTP (amb reintents automatics per a errors transitoris)
# ---------------------------------------------------------------------------
def _request(method, url, session=None, **kwargs):
    requester = session or requests
    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = requester.request(method, url, timeout=30, **kwargs)
            if resp.status_code in (502, 503, 504):
                cos = (resp.text or "").strip()[:300]
                raise requests.exceptions.HTTPError(
                    f"{resp.status_code} transitori | cos: {cos}", response=resp
                )
            resp.raise_for_status()
            return resp
        except (requests.exceptions.HTTPError, requests.exceptions.ConnectionError,
                requests.exceptions.Timeout) as e:
            last_error = e
            status = getattr(getattr(e, "response", None), "status_code", None)
            if status is not None and status not in (502, 503, 504):
                raise
            if attempt < MAX_RETRIES:
                wait = min(RETRY_BACKOFF_SECONDS * attempt, RETRY_BACKOFF_MAX)
                print(f"    (avis: {url} — {e} — reintent {attempt}/{MAX_RETRIES} en {wait}s)")
                time.sleep(wait)
    raise last_error


def _ajax_headers(codigo):
    return {
        "User-Agent": BROWSER_USER_AGENT,
        "X-Requested-With": "XMLHttpRequest",
        "Content-Type": "application/x-www-form-urlencoded; charset=UTF-8",
        "Referer": f"{BASE_URL}/modulos/clasif/clasificaciones.php?codigo={codigo}",
        "Origin": BASE_URL,
    }


def fetch_calendar_html():
    resp = _request("GET", CALENDAR_URL, session=_session, headers=HTML_HEADERS)
    return resp.text


def get_dorsales(codigo):
    resp = _request(
        "POST", DORSALES_URL, session=_session,
        data={"prueba": codigo, "subconjunto": SUBCONJUNTO},
        headers=_ajax_headers(codigo),
    )
    return resp.json().get("dorsales", [])


def get_tiempos(codigo):
    resp = _request(
        "POST", TIEMPOS_URL, session=_session,
        data={"prueba": codigo, "subconjunto": SUBCONJUNTO},
        headers=_ajax_headers(codigo),
    )
    return resp.json().get("tiempos", [])


# ---------------------------------------------------------------------------
# PARSING DEL CALENDARI (HTML estatic -> llista d'events)
# ---------------------------------------------------------------------------
def _text(el):
    return el.get_text(strip=True) if el else None


def _build_data(dia, mes, anio):
    if not (dia and mes and anio):
        return None
    mes_num = _MESOS.get(mes.strip().upper())
    if not mes_num:
        return None
    return f"{anio.strip()}-{mes_num}-{dia.strip().zfill(2)}"


def _codigo_from_href(href):
    if not href or "codigo=" not in href:
        return None
    return href.split("codigo=")[-1].split("&")[0].strip()


def parse_calendar_html(html):
    """Parseja l'HTML estatic de /clasificaciones/ i torna una llista
    d'events (cada un amb la seva llista de modalitats/codigo). Funcio pura
    -> facil de provar amb un fragment d'HTML fabricat a ma, sense xarxa."""
    soup = BeautifulSoup(html, "html.parser")
    events = []
    for item in soup.select(".crono-item"):
        dia = _text(item.select_one(".date-dia"))
        mes = _text(item.select_one(".date-mes"))
        anio = _text(item.select_one(".date-anio"))
        nom_cursa = _text(item.select_one(".crono-nombre"))

        ubicacio = item.select_one(".crono-ubicacion")
        municipi = _text(ubicacio.select_one("strong")) if ubicacio else None
        comarca_provincia = _text(ubicacio.select_one("span")) if ubicacio else None
        if comarca_provincia:
            comarca_provincia = comarca_provincia.strip("() ")

        modalitats = []
        for a in item.select("a.prueba-btn"):
            codigo = _codigo_from_href(a.get("href"))
            if codigo:
                modalitats.append({"codigo": codigo, "nom": _text(a)})

        events.append({
            # No hi ha cap identificador unic d'esdeveniment al calendari
            # (nomes "codigo" per modalitat) — fem servir el primer codigo
            # de l'esdeveniment com a id estable per detectar duplicats.
            "event_id": modalitats[0]["codigo"] if modalitats else None,
            "data": _build_data(dia, mes, anio),
            "nom_cursa": nom_cursa,
            "municipi": municipi,
            "comarca_provincia": comarca_provincia,
            "modalitats": modalitats,
        })
    return events


def fetch_calendar_events():
    html = fetch_calendar_html()
    events = parse_calendar_html(html)
    print(f"  (calendari: {len(events)} esdeveniments trobats)")
    return events


# ---------------------------------------------------------------------------
# TRANSFORMACIO A TAULA (facil de provar sense xarxa, amb fixtures a ma)
# ---------------------------------------------------------------------------
def build_classification(dorsales, tiempos):
    """Combina la llista d'inscrits (dorsales) amb els temps enregistrats
    (tiempos) i calcula les posicions (general, per sexe, per categoria)
    igual que fa el Javascript de la pagina — ordenant per temps ascendent
    — ja que l'API mateixa mai no torna "posicion" (sempre ve null)."""
    tiempos_by_dorsal = {t.get("dorsal"): t for t in tiempos}

    combinat = []
    for d in dorsales:
        t = tiempos_by_dorsal.get(d.get("dorsal"))
        temps = (t.get("tiempo") if t else None) or None
        estat_brut = (d.get("estado") or "").strip()
        combinat.append({
            "dorsal": d.get("dorsal"),
            "nom": d.get("nombre"),
            "cognoms": d.get("apellidos"),
            "sexo": d.get("sexo"),
            "club": d.get("club"),
            "categoria": d.get("categoria"),
            "local": d.get("local"),
            "temps": temps,
            "inici": t.get("inicio") if t else None,
            "fi": t.get("fin") if t else None,
            "estat": estat_brut if estat_brut else ("finished" if temps else "sense_temps"),
            "pos_general": None,
            "pos_genere": None,
            "pos_categoria": None,
        })

    finishers = [r for r in combinat if r["temps"]]
    # Els temps venen com "HH:MM:SS.mmm" amb amplada fixa i zero-padding, aixi
    # que ordenar-los com a text (lexicografic) dona el mateix resultat que
    # ordenar-los cronologicament.
    finishers.sort(key=lambda r: r["temps"])
    for i, r in enumerate(finishers, 1):
        r["pos_general"] = i

    for sexo in {r["sexo"] for r in finishers}:
        grup = sorted((r for r in finishers if r["sexo"] == sexo), key=lambda r: r["temps"])
        for i, r in enumerate(grup, 1):
            r["pos_genere"] = i

    for categoria in {r["categoria"] for r in finishers}:
        grup = sorted((r for r in finishers if r["categoria"] == categoria), key=lambda r: r["temps"])
        for i, r in enumerate(grup, 1):
            r["pos_categoria"] = i

    return combinat


def gender_status_breakdown(classification):
    """Compta, per (estat, sexe), quants corredors hi ha. Ho fem generic (no
    assumim quins valors d'"estat" existeixen mes enlla de "finished" /
    "sense_temps" — p.ex. "RET" o qualsevol altre codi que aparegui a les
    dades reals) perque no hem pogut confirmar tot el vocabulari possible
    nomes amb els exemples que hem vist."""
    counts = {}
    for r in classification:
        status = r.get("estat") or "desconegut"
        sexo = r.get("sexo") or "desconegut"
        counts.setdefault(status, {}).setdefault(sexo, 0)
        counts[status][sexo] += 1
    return counts


def _sexo_sufix(sexo):
    if sexo == "M":
        return "h"
    if sexo == "F":
        return "d"
    return sexo


def modalitat_row(event, modalitat, classification):
    row = {
        "event_id": event.get("event_id"),
        "nom_cursa": event.get("nom_cursa"),
        "data": event.get("data"),
        "municipi": event.get("municipi"),
        "comarca_provincia": event.get("comarca_provincia"),
        "codigo": modalitat.get("codigo"),
        "modalitat_nom": modalitat.get("nom"),
        "total_inscrits": len(classification),
        "total_classificats": sum(1 for r in classification if r["temps"]),
    }
    breakdown = gender_status_breakdown(classification)
    for status, per_sexo in breakdown.items():
        row[f"{status}_total"] = sum(per_sexo.values())
        for sexo, n in per_sexo.items():
            row[f"{status}_{_sexo_sufix(sexo)}"] = n
    return row


def runner_row(event, modalitat, r):
    return {
        "event_id": event.get("event_id"),
        "nom_cursa": event.get("nom_cursa"),
        "data": event.get("data"),
        "codigo": modalitat.get("codigo"),
        "modalitat_nom": modalitat.get("nom"),
        "dorsal": r.get("dorsal"),
        "nom": r.get("nom"),
        "cognoms": r.get("cognoms"),
        "sexo": r.get("sexo"),
        "club": r.get("club"),
        "categoria": r.get("categoria"),
        "local": r.get("local"),
        "estat": r.get("estat"),
        "temps": r.get("temps"),
        "inici": r.get("inici"),
        "fi": r.get("fi"),
        "pos_general": r.get("pos_general"),
        "pos_genere": r.get("pos_genere"),
        "pos_categoria": r.get("pos_categoria"),
    }


# ---------------------------------------------------------------------------
# ORQUESTRACIO
# ---------------------------------------------------------------------------
def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    already_done = _load_checkpoint() if RESUME else 0
    curses_rows = _load_csv(CURSES_CSV) if RESUME else []
    classif_rows = _load_csv(CLASSIF_CSV) if RESUME else []

    if RESUME and curses_rows:
        events_al_csv = len({row.get("event_id") for row in curses_rows if row.get("event_id")})
        if events_al_csv > already_done:
            print(f"AVIS: el checkpoint deia {already_done} esdeveniments fets, "
                  f"pero {CURSES_CSV} ja en te {events_al_csv} de diferents "
                  f"— s'utilitza {events_al_csv} per no repetir ni duplicar "
                  f"feina que ja estava feta.")
            already_done = events_al_csv
            _save_checkpoint(already_done)

    if already_done:
        print(f"Reprenent des de l'esdeveniment {already_done + 1} "
              f"(ja hi havia {already_done} processats — "
              f"{len(curses_rows)} files a {CURSES_CSV}, "
              f"{len(classif_rows)} a {CLASSIF_CSV}).")
        print("(Si vols començar de zero, esborra la carpeta "
              f"'{OUTPUT_DIR}' o posa RESUME = False.)\n")

    try:
        _run(curses_rows, classif_rows, already_done)
    except KeyboardInterrupt:
        print("\nInterromput manualment. Desant el que hi ha fins ara...")
    except Exception as e:
        print(f"\nS'ha aturat per un error persistent: {e}")
        print("No es perd res del que ja s'havia processat (es desa igualment "
              "mes avall). Torna-ho a provar mes tard executant main() de nou "
              "— continuara des d'on ho ha deixat.")
    finally:
        _write_csv(CURSES_CSV, curses_rows)
        _write_csv(CLASSIF_CSV, classif_rows)
        print(f"\nCurses/modalitats: {len(curses_rows)} files -> {CURSES_CSV}")
        print(f"Classificacions: {len(classif_rows)} files -> {CLASSIF_CSV}")

    return curses_rows, classif_rows


def _run(curses_rows, classif_rows, already_done=0):
    if MAX_EVENTS and MAX_EVENTS <= already_done:
        print(f"AVIS: MAX_EVENTS ({MAX_EVENTS}) es mes petit o igual que els "
              f"esdeveniments ja processats ({already_done}) — no hi ha res "
              f"nou a fer amb aquest limit. Puja MAX_EVENTS (o posa'l a None "
              f"per no limitar) i torna a executar main() per continuar.")
        return already_done

    events = fetch_calendar_events()
    if MAX_EVENTS:
        events = events[:MAX_EVENTS]

    idx = already_done
    for idx, event in enumerate(events, 1):
        if idx <= already_done:
            continue

        try:
            _process_event(event, idx, curses_rows, classif_rows)
        except Exception as e:
            print(f"  ERROR processant '{event.get('nom_cursa')}': {e} — el salto")

        if idx % SAVE_EVERY_N_EVENTS == 0:
            _write_csv(CURSES_CSV, curses_rows)
            _write_csv(CLASSIF_CSV, classif_rows)
            _save_checkpoint(idx)
            print(f"  (progres desat: {idx} esdeveniments processats fins ara)")

    _save_checkpoint(max(idx, already_done))
    return max(idx, already_done)


def _process_event(event, idx, curses_rows, classif_rows):
    modalitats = event.get("modalitats") or []
    print(f"[{idx}] {event.get('nom_cursa')} ({event.get('data')}) — "
          f"{len(modalitats)} modalitat(s)")

    for modalitat in modalitats:
        codigo = modalitat.get("codigo")
        if not codigo:
            continue

        try:
            dorsales = get_dorsales(codigo)
            tiempos = get_tiempos(codigo)
        except Exception as e:
            print(f"    ERROR modalitat {modalitat.get('nom')!r} (codigo={codigo}): {e}")
            continue

        classification = build_classification(dorsales, tiempos)
        n_classificats = sum(1 for r in classification if r["temps"])
        print(f"    {modalitat.get('nom')}: {len(classification)} inscrits, "
              f"{n_classificats} classificats")

        curses_rows.append(modalitat_row(event, modalitat, classification))
        for r in classification:
            classif_rows.append(runner_row(event, modalitat, r))

        time.sleep(SLEEP_BETWEEN_REQUESTS)


def _write_csv(path, rows):
    if not rows:
        return

    if RESUME and os.path.exists(path):
        existent = _load_csv(path)
        if len(existent) > len(rows):
            path_revisio = path + ".NOMES_LECTURA_revisa.csv"
            fieldnames = sorted({k for row in rows for k in row.keys()})
            with open(path_revisio, "w", newline="", encoding="utf-8-sig") as f:
                writer = csv.DictWriter(f, fieldnames=fieldnames)
                writer.writeheader()
                writer.writerows(rows)
            print(f"AVIS IMPORTANT: {path} ja te {len(existent)} files i "
                  f"anavem a escriure'n nomes {len(rows)} — aixo sembla un "
                  f"error (es perdria feina feta), aixi que NO he tocat "
                  f"{path}. He desat les {len(rows)} files noves a "
                  f"'{path_revisio}' perque ho revisis tu abans de decidir "
                  f"que fer.")
            return

    fieldnames = sorted({k for row in rows for k in row.keys()})
    with open(path, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def _load_csv(path):
    if not os.path.exists(path):
        return []
    with open(path, newline="", encoding="utf-8-sig") as f:
        return list(csv.DictReader(f))


def _load_checkpoint():
    if not os.path.exists(CHECKPOINT_FILE):
        return 0
    with open(CHECKPOINT_FILE, encoding="utf-8") as f:
        return json.load(f).get("completed_events", 0)


def _save_checkpoint(completed_events):
    with open(CHECKPOINT_FILE, "w", encoding="utf-8") as f:
        json.dump({"completed_events": completed_events}, f)


if __name__ == "__main__":
    main()

Reprenent des de l'esdeveniment 148 (ja hi havia 147 processats — 314 files a buscametas_output\curses.csv, 74374 a buscametas_output\classificacions.csv).
(Si vols començar de zero, esborra la carpeta 'buscametas_output' o posa RESUME = False.)

  (calendari: 147 esdeveniments trobats)

Curses/modalitats: 314 files -> buscametas_output\curses.csv
Classificacions: 74374 files -> buscametas_output\classificacions.csv
